# SimPipeline live console

Starts the console server (`python -m console`) in the background and opens it. The console runs one simulation: press play, watch online CTR, retrain, roll back, change pipeline settings and handle incidents.

**How to run:** run all cells. In Colab the console opens in a new browser tab through Colab's port proxy; elsewhere the last cell prints a local URL.
- If the notebook is inside the repo it uses that code; otherwise the first cell downloads the repo (`REPO_REF` picks the branch) and installs missing packages (torch, fastapi, uvicorn).
- The simulation keeps running in the server process while this kernel is alive. Run the last cell to stop it.


In [ ]:
import importlib
import importlib.util
import inspect
import io
import os
import shutil
import subprocess
import sys
import urllib.request
import zipfile
from pathlib import Path

REPO_URL = "https://github.com/PrimozRavbar/SimPipelineExp"
REPO_REF = "main"   # branch, tag or commit to download when the notebook is not inside the repo
                    # (before the console is merged, use "claude/console-live")


def is_repo(path):
    return (path / "ml_system").is_dir() and (path / "world").is_dir()


def download_zip(ref, target):
    """Fetch the repo as a zip (no git needed) and unpack it into target."""
    with urllib.request.urlopen(f"{REPO_URL}/archive/{ref}.zip") as response:
        archive = zipfile.ZipFile(io.BytesIO(response.read()))
    top = archive.namelist()[0].split("/")[0]
    staging = target.parent / (target.name + "-download")
    shutil.rmtree(staging, ignore_errors=True)
    archive.extractall(staging)
    shutil.rmtree(target, ignore_errors=True)
    shutil.move(str(staging / top), str(target))
    shutil.rmtree(staging, ignore_errors=True)


# 1. Find the code: the repo this notebook is in, or a copy next to it.
ROOT = Path.cwd()
if not is_repo(ROOT):
    ROOT = Path.cwd() / "SimPipelineExp"
    if (ROOT / ".git").exists():
        # A clone from an earlier run: update it to REPO_REF.
        subprocess.run(["git", "-C", str(ROOT), "fetch", "-q", "origin", REPO_REF], check=True)
        subprocess.run(["git", "-C", str(ROOT), "checkout", "-q", "--detach", "FETCH_HEAD"], check=True)
    elif shutil.which("git") and not ROOT.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL + ".git", str(ROOT)], check=True)
        subprocess.run(["git", "-C", str(ROOT), "fetch", "-q", "origin", REPO_REF], check=True)
        subprocess.run(["git", "-C", str(ROOT), "checkout", "-q", "--detach", "FETCH_HEAD"], check=True)
    else:
        print(f"Downloading {REPO_REF} as a zip into {ROOT} ...")
        download_zip(REPO_REF, ROOT)
    os.chdir(ROOT)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# 2. Install missing packages into this kernel's Python.
missing = [name for name in ("torch", "pandas", "numpy", "fastapi", "uvicorn")
           if importlib.util.find_spec(name) is None]
if missing:
    print("Installing", ", ".join(missing), "(torch is large: the first run takes a few minutes) ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
    importlib.invalidate_caches()

# 3. Forget repo modules imported earlier in this kernel, so the code above is used.
for name in list(sys.modules):
    if name.split(".")[0] in ("simulation", "world", "ml_system", "dashboard", "console"):
        del sys.modules[name]
importlib.invalidate_caches()

# 4. Fail early, with a clear message, if the code is older than this notebook.
if not (ROOT / "console" / "server.py").exists():
    raise RuntimeError(
        f"The code in {ROOT} is older than this notebook. Update it (git pull), "
        "or delete that folder and rerun this cell to download a fresh copy."
    )

commit = ""
if shutil.which("git") and (ROOT / ".git").exists():
    commit = subprocess.run(["git", "-C", str(ROOT), "rev-parse", "--short", "HEAD"],
                            capture_output=True, text=True).stdout.strip()
print("Repo:", ROOT, "| commit:", commit or f"{REPO_REF} (zip download)")


## Settings

In [ ]:
SEED = 42             # same seed + same actions at the same ticks -> same run
USERS_PER_TICK = 10   # distinct users served per tick
PORT = 8000
WORLD = "static"     # "static" or "dynamic" (tastes drift, users tire of repeats, new users join)
TASTE = "genre"      # "genre" or "rich" (tastes beyond genres, learned from the real ratings)
PRESET = "best"      # "best" (measured best setup) or "plain" (every model feature off)


## Start the server

In [ ]:
import json
import time
import urllib.request

def server_up():
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/api/state", timeout=1) as r:
            return json.load(r)
    except Exception:
        return None

if server_up():
    print("A console server is already running on port", PORT, "(run the last cell to stop it).")
else:
    log_file = open(ROOT / "console-server.log", "w")
    server = subprocess.Popen(
        [sys.executable, "-m", "console", "--port", str(PORT), "--host", "127.0.0.1",
         "--seed", str(SEED), "--users-per-tick", str(USERS_PER_TICK),
         "--world", WORLD, "--taste", TASTE, "--preset", PRESET],
        cwd=str(ROOT), stdout=log_file, stderr=subprocess.STDOUT,
    )
    print("Building the simulation (about 10-30 s) ...")
    for _ in range(240):
        if server_up() or server.poll() is not None:
            break
        time.sleep(0.5)
    state = server_up()
    if state is None:
        print(open(ROOT / "console-server.log").read())
        raise RuntimeError("The console server did not start; its log is above.")
    print("Console server running: tick", state["tick"], "| seed", state["seed"])


## Open the console

In [ ]:
try:
    from google.colab import output
    output.serve_kernel_port_as_window(PORT, anchor_text="Open the SimPipeline console")
except ImportError:
    url = f"http://localhost:{PORT}"
    print("Console:", url)
    try:
        from IPython.display import HTML, display
        display(HTML(f'<a href="{url}" target="_blank">Open the SimPipeline console</a>'))
    except ImportError:
        pass


## Stop the server

In [ ]:
try:
    server.terminate()
    server.wait(timeout=10)
    print("Console server stopped.")
except NameError:
    print("This kernel did not start a server.")
